# Phrase-level speculative decoding vs EAGLE-3 on Qwen3-4B

A **21.9 M** draft head predicts a whole 1–3 word phrase as one vocabulary entry;
SLEM expands it to target tokens by lookup; N candidates are merged into a prefix
tree and verified in **one** target forward pass. Output is greedy-identical up to
fp16 tie-breaking.

Both methods draft multiple tokens ahead. EAGLE-3 gets there by running its draft
model **autoregressively seven times** per verification pass; we get there in **one**
head forward, because a vocabulary entry already *is* ~2.46 tokens.

Compared against **EAGLE-3** (218.2 M, 9.95× larger) run through SafeAILab's own
code at a pinned commit — which reproduces DFlash's published Table 1 to two
decimals, so the baseline here is the real one.

| set | ours | speedup | tok/pass | EAGLE-3 | speedup | tok/pass | paired Δ ms/tok |
|---|---|---|---|---|---|---|---|
| MATH-500 | N16 | 2.037 | 2.620 | tree 16 | 2.061 | 3.035 | **+0.66 — tie** |
| GSM8K | N32 | 2.211 | 2.742 | tree 60 | 2.451 | 3.811 | +4.44 |
| MT-Bench | N16 | 1.402 | 1.672 | tree 60 | 2.005 | 3.324 | +14.03 |

One T4, fp16, batch 1, temperature 0, EOS-terminated with a 2048 cap. Same prompts
on both sides, loaded from one file.

**Prerequisite:** a trained head from `pipeline/1_train_head.py`. No weights are
shipped; stage 1 distils, builds the vocabulary and trains, and takes ~3 h on a T4.

In [ ]:
import os, subprocess, sys

# the repo root, whether this runs from notebook/ or from the root
ROOT = os.path.abspath("..") if os.path.isdir("../pipeline") else os.path.abspath(".")
PIPE = os.path.join(ROOT, "pipeline")
RES = os.path.join(ROOT, "results")
assert os.path.isdir(PIPE), "run this from the repo root or from notebook/"
print("root   ", ROOT)
print("stages ", sorted(os.listdir(PIPE)))
print("results", sorted(os.listdir(RES)))


def stage(name):
    """Each stage runs as its own PROCESS, not with %run.

    Stage 2 pins transformers==4.53.1 because the EAGLE commit it clones needs it,
    and a pin cannot take effect inside a kernel that has already imported
    transformers. Separate processes keep the two environments from colliding."""
    p = os.path.join(PIPE, name)
    print("=" * 70 + "\n" + name + "\n" + "=" * 70, flush=True)
    r = subprocess.run([sys.executable, p], cwd=ROOT)
    assert r.returncode == 0, "%s exited %d" % (name, r.returncode)

## 1. EAGLE-3

Pins `transformers==4.53.1`, clones SafeAILab/EAGLE at `cb7e0841fe0c`, and applies
the one patch the 4B head needs: Qwen3-4B's attention is wider than its hidden size
(32 × 128 = 4096 against 2560) and the official draft code derives head width from
hidden size alone, so it cannot load that head at all without honouring
`config.head_dim`.

Runs a **sanity gate first**: 6 MT-Bench prompts must clear 2.0 tok/pass, since
EAGLE-3 was trained on chat and the published acceptance length there is ~3.4. If
the gate fails the setup is wrong and the run aborts rather than reporting a number.

Also writes `results/e89_prompts.json` — the prompt set stage 2 loads.

In [ ]:
stage("2_bench_eagle3.py")

## 2. Ours

**Loads** the prompt file written above and asserts its sha, rather than re-drawing
the pool — an earlier version re-derived it per script and one differing constant
silently changed which prompts were selected. Rebuilds the vocabulary from the
distilled corpus and asserts it matches the head checkpoint on all 80,000 entries;
if that fails, the head's output rows mean something other than what we would draft.

In [ ]:
stage("3_bench_ours.py")

## 3. The comparison

Paired per prompt on **ms per generated token**. A speedup is a ratio to its own
baseline and the two runs use different baselines — ours measures 63.7 ms/tok
against EAGLE's `naivegenerate` at 59.6, consistently, which inflates our speedup
column by ~7%. Milliseconds per token carries no baseline, so this compares the two
methods and not the two Python loops.

Pairing is **checked**: both sides are greedy at temperature 0 on the same weights,
so baseline token counts must agree per prompt.

In [ ]:
import numpy as np
import pandas as pd

BOOT = 4000


def load(name):
    p = os.path.join(RES, name)
    return pd.read_csv(p) if os.path.exists(p) else None


E, O = load("e89_pairs.csv"), load("e91_pairs.csv")
assert E is not None, "results/e89_pairs.csv missing -- run stage 1 above"
assert O is not None, (
    "results/e91_pairs.csv missing -- run stage 2 above. (The shipped results/ has"
    " EAGLE's per-prompt records but not ours: the run that produced the published"
    " table lost them to a write bug. results/README.md records it.)")


def mspt(d, tok="arm_tok"):
    return 1000.0 * d["arm_secs"] / d[tok].clip(lower=1)


def boot_mean_ci(x):
    x = np.asarray(x, float)
    if x.size < 2:
        return float("nan"), float("nan")
    r = np.random.default_rng(7)
    b = np.array([r.choice(x, x.size, replace=True).mean() for _ in range(BOOT)])
    return float(np.percentile(b, 2.5)), float(np.percentile(b, 97.5))


print("%-12s %-16s %-16s %-13s %-22s %s"
      % ("set", "ours (best)", "EAGLE-3 (best)", "delta ms/tok", "95% CI", "verdict"))
for ts in ("math500", "gsm8k-test", "mtbench"):
    o, e = O[O["test"] == ts], E[E["test"] == ts]
    if o.empty or e.empty:
        continue
    # "best" = lowest mean ms/token, the same rule on both sides
    bo = min(o["arm"].unique(), key=lambda a: mspt(o[o["arm"] == a]).mean())
    be = min(e["arm"].unique(), key=lambda a: mspt(e[e["arm"] == a]).mean())
    ou = o[o["arm"] == bo].set_index("seq")
    ea = e[e["arm"] == be].set_index("seq")
    ks = sorted(set(ou.index) & set(ea.index))

    # prompt identity: same question, same greedy decode -> same baseline length
    dif = np.array([abs(int(ou.loc[k, "base_tok"]) - int(ea.loc[k, "base_tok"]))
                    for k in ks], dtype=float)
    mn = float(np.mean([int(ea.loc[k, "base_tok"]) for k in ks]))
    exact = int((dif == 0).sum())
    if exact < 0.6 * len(ks) or dif.mean() > 0.08 * mn:
        print("  %-12s SKIPPED: only %d/%d baselines exact, mean |diff| %.1f tok --"
              " too far apart to be fp16 tie-breaking, so these are not the same"
              " prompts" % (ts, exact, len(ks), dif.mean()))
        continue

    u = mspt(ou.loc[ks]).values
    v = mspt(ea.loc[ks]).values
    d = u - v
    lo, hi = boot_mean_ci(d)
    print("  %-12s %-16s %-16s %+-13.2f [%+.2f, %+.2f]%s   n=%d  (%d/%d exact)"
          % (ts, "%s %.1f" % (bo, u.mean()), "%s %.1f" % (be, v.mean()),
             d.mean(), lo, hi,
             "  OURS faster" if hi < 0 else
             ("  EAGLE faster" if lo > 0 else "  tie        "),
             len(ks), exact, len(ks)))

In [ ]:
import matplotlib.pyplot as plt

SETS = [s for s in ("math500", "gsm8k-test", "mtbench")
        if (O["test"] == s).any() and (E["test"] == s).any()]
ours = [mspt(O[(O["test"] == s)]).groupby(O[O["test"] == s]["arm"]).mean().min()
        for s in SETS]
eagl = [mspt(E[(E["test"] == s)]).groupby(E[E["test"] == s]["arm"]).mean().min()
        for s in SETS]
base = [1000.0 * E[E["test"] == s]["base_secs"].sum()
        / max(E[E["test"] == s]["base_tok"].sum(), 1) for s in SETS]

x = np.arange(len(SETS))
w = 0.26
fig, ax = plt.subplots(figsize=(7.5, 4.0))
ax.bar(x - w, base, w, label="baseline (no drafting)", color="#8A93A6")
ax.bar(x, eagl, w, label="EAGLE-3 (218.2 M)", color="#4C78A8")
ax.bar(x + w, ours, w, label="ours (21.9 M)", color="#F58518")
for xi, (b, e, o) in enumerate(zip(base, eagl, ours)):
    for dx, val in ((-w, b), (0, e), (w, o)):
        ax.text(xi + dx, val + 0.6, "%.1f" % val, ha="center", fontsize=8)
ax.set_xticks(x)
ax.set_xticklabels(SETS)
ax.set_ylabel("ms per generated token  (lower is better)")
ax.set_title("Qwen3-4B on one T4, temperature 0, EOS-terminated")
ax.legend(frameon=False, fontsize=9)
ax.spines[["top", "right"]].set_visible(False)
ax.set_axisbelow(True)
ax.grid(axis="y", color="#C4CAD4", lw=0.6)
plt.tight_layout()
plt.savefig(os.path.join(RES, "mspt.png"), dpi=160)
plt.show()

## What this shows, and what it does not

**Level with EAGLE-3 on MATH-500, within 18% on GSM8K, with a draft head 10×
smaller trained on ~3 M tokens.** Not that it beats EAGLE-3.

- **We lose on drafting, not on cost.** Per verification pass on GSM8K ours costs
  **79.0 ms against a 63.7 ms baseline (+24%)** and EAGLE-3 costs **92.7 against
  59.6 (+56%)** — one head forward and a table lookup against seven autoregressive
  draft forwards. Our pass is 15% cheaper, we feed fewer nodes (46 against 60), and
  we still lose because we bank 2.742 tokens in it against 3.811.
- **MT-Bench is the generality bound.** The head has only ever seen maths. Boundary
  coverage there is still 1.000, so it is not that the vocabulary lacks entries —
  the phrases are the wrong ones.
- **The MATH-500 result splits.** Level on the typical prompt, 12% behind across the
  whole workload; the difference is entirely the long answers, and that set averages
  571 tokens.
- **Exact.** 98 divergences over 420 arm runs (3 arms x 140 prompts), 0 confident,
  worst logit margin 0.0625 against a 0.5 fp16 threshold — i.e. **0.57 per 1,000
  token positions**, the same rate EAGLE-3’s own implementation shows on these
  prompts. Every one of ours is a tie-break, not a misprediction.

### These are floor numbers

Three knobs decide the ceiling and none has been explored:

1. **Unit length.** `MAX_GRAM = 3`, mean expansion 2.462 target tokens, so tok/pass
   is capped at **3.46** however good the head gets. EAGLE already exceeds that at
   3.811, so no amount of training closes the GSM8K gap at gram 3. Raising it to
   5–6 is the only change that lifts the bound.
2. **Vocabulary size.** The 80,000 cap binds, not the parameter budget — which would
   allow 150,282. The head is 21.9 M because of a number that was chosen, not found.
3. **Training data.** The ladder has not bent across a 6.5× increase; every rung
   still paid.

And N has not peaked on GSM8K: 1.985 → 2.108 → 2.211 across N = 8/16/32, still
climbing. MATH-500 and MT-Bench both turn over at N16.

### One hardware caveat that applies to every speedup above

**T4 only.** The decode loop is CPU-dispatch-bound — the target forward is 92.6% of
a pass and essentially all of that is the CPU issuing work. The same pipeline on an
A5000 measured 1.061× with *better* acceptance (tok/pass 2.695), because a faster
card shrinks the GPU work and exposes the Python a T4 was slow enough to hide.
`tok/pass` is hardware-independent and transfers; the speedups do not.